        # Análisis de sensibilidad básico

        **Modelación y Simulación Computacional** · Maestría en Ingeniería ·
        Universidad de Sucre · periodo 2026-2

        **Unidad 3.** Simulación de sistemas y análisis de escenarios ·
        **Subtema del plan 3.4**

        Autor, Prof. Daniel Otero Meza, Ing., Ph.D.

        <!-- ENLACE_COLAB -->
        [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/<usuario>/<repositorio>/blob/main/03_cuadernos/Unidad3/U3_04_analisis_de_sensibilidad.ipynb)

        El cuaderno anterior entregó una distribución de la respuesta y queda por
responder de dónde viene esa dispersión. Si un solo parámetro explica el
ochenta por ciento de la varianza, medirlo mejor reduce la incertidumbre
del pronóstico y afinar los demás no sirve de nada. Aquí se recorren los
tres enfoques de la Tabla 3.4 del libro, la elasticidad local, el
tamizado de Morris y los índices de Sobol', cada uno con su verificación.

        ## Objetivos de aprendizaje

        Al terminar este cuaderno el estudiante debe ser capaz de

        1. Calcular elasticidades por diferencias centradas según la Definición 3.9 y verificarlas con las identidades que impone la estructura del modelo.
2. Construir un diagrama de tornado y explicar por qué ordena los factores de manera distinta que la elasticidad.
3. Implementar el tamizado de Morris del Algoritmo 3.4 y verificarlo con un modelo lineal de efectos conocidos.
4. Estimar los índices de Sobol' de primer orden y totales con los estimadores de Saltelli y de Jansen, y verificarlos con la función de prueba de Ishigami.
5. Traducir la jerarquía de factores en una recomendación de medición, de calibración o de simplificación del modelo.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de modo que
el cuaderno abre igual en Google Colab y en JupyterLab. La segunda fija la
semilla del curso, la paleta del libro y las funciones auxiliares. La semilla
vale 20262 y ningún resultado depende de una ejecución concreta.

In [ ]:
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict) -> None:
    """Instala solo los paquetes que no estén disponibles."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

COLORES = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.figsize": (9.0, 4.4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "legend.frameon": False})

trapecio = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def carpeta_datos() -> Path:
    """Ubica la carpeta datos sin usar rutas absolutas.

    Busca hacia arriba desde el directorio de trabajo, de modo que funcione
    tanto en el repositorio como en una sesión de Colab donde el cuaderno se
    abre suelto. Si no la encuentra, la crea junto al cuaderno.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        if (candidata / "datos").is_dir():
            return candidata / "datos"
    destino = base / "datos"
    destino.mkdir(exist_ok=True)
    return destino


def leer_datos(nombre: str, respaldo) -> pd.DataFrame:
    """Lee un archivo de datos y lo reconstruye si no está disponible.

    El argumento respaldo es una función sin argumentos que devuelve el
    mismo cuadro de datos, construido con las cifras publicadas en el libro.
    Así el cuaderno nunca depende de una descarga.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        ruta = candidata / "datos" / nombre
        if ruta.exists():
            return pd.read_csv(ruta)
    tabla = respaldo()
    tabla.to_csv(carpeta_datos() / nombre, index=False)
    return tabla


def comparar(etiqueta: str, calculado: float, libro: float,
             tol: float, unidad: str = "") -> bool:
    """Imprime y verifica un valor calculado frente al que publica el libro."""
    dif = abs(calculado - libro)
    ok = dif <= tol
    marca = "coincide" if ok else "NO coincide"
    print(f"{etiqueta:<46s} calculado {calculado:>14.6g} {unidad:<12s}"
          f" libro {libro:>12.6g}   {marca}")
    return ok


print("semilla del curso", SEMILLA)

In [ ]:
def respaldo_valores_libro() -> pd.DataFrame:
    """Cifras publicadas en el capítulo 3, transcritas del libro."""
    filas = [
    ("colebrook_velocidad", 1.6977, "m/s", "Ejemplo 3.1"),
    ("colebrook_reynolds", 507267.0, "adimensional", "Ejemplo 3.1"),
    ("colebrook_rugosidad_relativa", 0.0008667, "adimensional", "Ejemplo 3.1"),
    ("colebrook_factor_friccion", 0.0196228, "adimensional", "Ejemplo 3.1"),
    ("colebrook_perdida_carga", 8.167, "m", "Ejemplo 3.1"),
    ("colebrook_swamee_jain", 0.019742, "adimensional", "Ejemplo 3.1"),
    ("colebrook_orden_newton", 2.0, "adimensional", "Ejemplo 3.1"),
    ("lagunas_perfil_1", 142.42, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_2", 83.4, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_3", 41.5, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_4", 17.65, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_5", 7.33, "mg/L", "seccion 3.1.2"),
    ("lagunas_remocion", 97.07, "por ciento", "seccion 3.1.2"),
    ("lagunas_retencion", 8.33, "d", "seccion 3.1.2"),
    ("lagunas_carga_afluente", 300000.0, "mg/d", "seccion 3.1.2"),
    ("lagunas_carga_efluente", 8792.84, "mg/d", "seccion 3.1.2"),
    ("lagunas_consumo", 291207.16, "mg/d", "seccion 3.1.2"),
    ("fermentador_tiempo_25C", 17.0604614, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_30C", 10.9186953, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_35C", 7.5824273, "h", "Ejemplo 3.2"),
    ("fermentador_invariante", 12.5, "g/L", "Ejemplo 3.2"),
    ("fermentador_mumax_25C", 0.192, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_30C", 0.3, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_35C", 0.432, "1/h", "Ejemplo 3.2"),
    ("fermentador_evaluaciones", 584.0, "evaluaciones", "Ejemplo 3.2"),
    ("tolerancia_tiempo_rtol3", 10.9028, "h", "seccion 3.2.1"),
    ("tolerancia_error_rtol3", 0.00146, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol3", 44.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol6", 1.85e-07, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol6", 194.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol9", 2.45e-10, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol9", 584.0, "evaluaciones", "seccion 3.2.1"),
    ("circuito_autovalor_rapido", -1005.0002, "1/s", "Ejemplo 3.3"),
    ("circuito_autovalor_lento", -0.0497512, "1/s", "Ejemplo 3.3"),
    ("circuito_razon_rigidez", 20200.0, "adimensional", "Ejemplo 3.3"),
    ("circuito_pasos_rk45", 30376.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_rk45", 212576.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_pasos_bdf", 144.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_bdf", 292.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_paso_medio_rk45", 0.003292, "s", "Ejemplo 3.3"),
    ("circuito_tau_rapida", 0.000995, "s", "Ejemplo 3.3"),
    ("circuito_tau_lenta", 20.1, "s", "Ejemplo 3.3"),
    ("circuito_error_rk45", 9.1e-07, "adimensional", "Ejemplo 3.3"),
    ("circuito_error_bdf", 1.1e-06, "adimensional", "Ejemplo 3.3"),
    ("circuito_producto_h_lambda", 3.31, "adimensional", "Ejemplo 3.3"),
    ("rio_peclet_celda", 0.583, "adimensional", "Ejemplo 3.4"),
    ("rio_pico_analitico", 1.8655, "mg/L", "Ejemplo 3.4"),
    ("rio_abscisa_pico", 2460.0, "m", "Ejemplo 3.4"),
    ("rio_error_maximo", 7.18e-06, "kg/m3", "Ejemplo 3.4"),
    ("rio_masa_remanente", 24.740935, "kg", "Ejemplo 3.4"),
    ("rio_orden_observado", 2.0, "adimensional", "Ejemplo 3.4"),
    ("rio_paso_difusion", 16.67, "s", "seccion 3.3.2"),
    ("rio_paso_adveccion", 57.14, "s", "seccion 3.3.2"),
    ("rio_error_explicito_d045", 6.3e-05, "kg/m3", "Ejemplo 3.4"),
    ("riego_frontera_bruto_p045_d45", 393.8, "mm", "Ejemplo 3.5"),
    ("riego_frontera_bruto_p085_d65", 243.8, "mm", "Ejemplo 3.5"),
    ("riego_deficit_p085_d65", 21.5, "por ciento", "Ejemplo 3.5"),
    ("riego_no_dominadas_clima_normal", 5.0, "alternativas", "Ejemplo 3.5"),
    ("biogas_desviacion_replicas_mc", 0.933, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion_replicas_lhs", 0.112, "kW h/d", "Ejemplo 3.6"),
    ("biogas_reduccion_varianza", 69.0, "veces", "Ejemplo 3.6"),
    ("riego_agua_aprovechable", 126.0, "mm", "Ejemplo 3.5"),
    ("biogas_media", 92.34, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion", 21.22, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p05", 61.71, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p50", 90.05, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p95", 130.62, "kW h/d", "Ejemplo 3.6"),
    ("biogas_excedencia_110", 0.1945, "adimensional", "Ejemplo 3.6"),
    ("biogas_error_estandar", 0.15, "kW h/d", "Ejemplo 3.6"),
    ("biogas_valores_centrales", 91.53, "kW h/d", "Ejemplo 3.6"),
    ("lcoe_crf", 0.101806, "1/a", "Ejemplo 3.7"),
    ("lcoe_factor_degradacion", 0.931205, "adimensional", "Ejemplo 3.7"),
    ("lcoe_produccion_especifica", 1325.6, "kW h/(kW a)", "Ejemplo 3.7"),
    ("lcoe_nominal", 0.083523, "USD/(kW h)", "Ejemplo 3.7"),
    ("lcoe_elasticidad_inversion", 0.87355, "adimensional", "Ejemplo 3.7"),
    ("lcoe_elasticidad_tasa", 0.637, "adimensional", "Ejemplo 3.7"),
    ("lcoe_amplitud_tasa", 35.8, "por ciento", "Ejemplo 3.7"),
    ("lcoe_amplitud_irradiacion", 16.1, "por ciento", "Ejemplo 3.7"),
    ("ishigami_s1", 0.3138, "adimensional", "seccion 3.6.2"),
    ("ishigami_s2", 0.4423, "adimensional", "seccion 3.6.2"),
    ("ishigami_s3", -0.0001, "adimensional", "seccion 3.6.2"),
    ("ishigami_st3", 0.2436, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_s_B0", 0.616, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_suma_primer_orden", 0.985, "adimensional", "seccion 3.6.2"),
    ("morris_evaluaciones", 210.0, "evaluaciones", "seccion 3.6.2"),
    ]
    return pd.DataFrame(filas, columns=["clave", "valor", "unidad", "referencia"])


LIBRO = leer_datos("valores_libro_cap3.csv",
                   respaldo_valores_libro).set_index("clave")["valor"]
print(f"cifras del libro disponibles, {LIBRO.size} registros")

## 1. Sensibilidad local y elasticidad

La sensibilidad local es la derivada parcial de la respuesta respecto de
una entrada, evaluada en el punto nominal, y su cálculo por diferencias
centradas cuesta dos evaluaciones por entrada. Sus unidades son las de la
respuesta divididas por las de la entrada, detalle que impide usarla como
criterio de jerarquía. La Definición 3.9 del libro resuelve el problema
con la elasticidad, que indica el cambio porcentual de la respuesta que
produce un cambio del uno por ciento en la entrada.

El Ejemplo 3.7 instala un sistema fotovoltaico de 250 kW nominales en el
Caribe colombiano, con irradiación diaria de referencia de
5.0 kW h/(m² d), razón de desempeño de 0.78, inversión de 950 USD/kW,
operación y mantenimiento de 14 USD/(kW a), tasa de descuento de 0.09,
vida útil de 25 años y degradación anual de 0.006. El costo nivelado sale
de la Ecuación 3.24.

In [ ]:
def respaldo_factores_lcoe() -> pd.DataFrame:
    """Valores nominales y rangos plausibles del Ejemplo 3.7."""
    filas = [
        ("Hs", "irradiacion_diaria", 5.0, 4.6, 5.4, "kW h/(m2 d)"),
        ("PR", "razon_de_desempeno", 0.78, 0.72, 0.83, "adimensional"),
        ("Cinv", "inversion_especifica", 950.0, 820.0, 1150.0, "USD/kW"),
        ("Com", "operacion_anual", 14.0, 10.0, 20.0, "USD/(kW a)"),
        ("i", "tasa_de_descuento", 0.09, 0.07, 0.12, "adimensional"),
        ("n", "vida_util", 25.0, 20.0, 30.0, "a"),
        ("delta", "degradacion_anual", 0.006, 0.004, 0.009, "1/a")]
    return pd.DataFrame(filas, columns=["factor", "descripcion", "nominal",
                                        "minimo", "maximo", "unidad"])


factores = leer_datos("factores_lcoe.csv", respaldo_factores_lcoe)
NOMINAL = dict(zip(factores["factor"], factores["nominal"]))
RANGOS = {f: (lo, hi) for f, lo, hi in zip(factores["factor"],
                                           factores["minimo"],
                                           factores["maximo"])}
print(factores.to_string(index=False))


def lcoe(Hs, PR, Cinv, Com, i, n, delta):
    """Costo nivelado de la energía en USD por kilovatio hora."""
    crf = i * (1 + i)**n / ((1 + i)**n - 1)
    degradacion = (1 - (1 - delta)**n) / (n * delta)
    return (Cinv * crf + Com) / (Hs * 365.0 * PR * degradacion)

In [ ]:
crf_nominal = (NOMINAL["i"] * (1 + NOMINAL["i"])**NOMINAL["n"]
               / ((1 + NOMINAL["i"])**NOMINAL["n"] - 1))
fd_nominal = ((1 - (1 - NOMINAL["delta"])**NOMINAL["n"])
              / (NOMINAL["n"] * NOMINAL["delta"]))
produccion = NOMINAL["Hs"] * 365.0 * NOMINAL["PR"] * fd_nominal
costo_nominal = lcoe(**NOMINAL)

ok = [comparar("factor de recuperación del capital", crf_nominal,
               LIBRO["lcoe_crf"], 5e-7, "1/a"),
      comparar("factor medio de degradación", fd_nominal,
               LIBRO["lcoe_factor_degradacion"], 5e-7, ""),
      comparar("producción específica anual", produccion,
               LIBRO["lcoe_produccion_especifica"], 5e-2, "kW h/(kW a)"),
      comparar("costo nivelado nominal", costo_nominal,
               LIBRO["lcoe_nominal"], 5e-7, "USD/(kW h)")]
assert all(ok), "el costo nivelado no reproduce el Ejemplo 3.7"

### Ejercicio 1

Complete el cálculo de las elasticidades por diferencias centradas de
paso relativo, tal como hace el Listado 3.9 del libro. Para cada factor,
perturbe su valor nominal en más y en menos un paso proporcional al
propio valor, evalúe el modelo en ambos puntos y adimensionalice el
cociente incremental con el valor nominal del factor y de la respuesta.
La celda de partida devuelve ceros.

In [ ]:
# COMPLETE: E_j = (modelo(x+h) - modelo(x-h))/(2h) * x0/y0 con h = paso*|x0|
REVISAR_ELAST = False


def elasticidades(modelo, nominal: dict, paso: float = 1e-4) -> dict:
    """Elasticidad de la respuesta frente a cada factor."""
    y0 = modelo(**nominal)
    resultado = {}
    for nombre_factor, x0 in nominal.items():
        resultado[nombre_factor] = 0.0      # marcador de posición
    return resultado

In [ ]:
elast = elasticidades(lcoe, NOMINAL)
tabla_elast = pd.DataFrame({"factor": list(elast), "elasticidad": list(elast.values())})
tabla_elast["magnitud"] = tabla_elast["elasticidad"].abs()
tabla_elast = tabla_elast.sort_values("magnitud", ascending=False)
print(tabla_elast[["factor", "elasticidad"]].to_string(
    index=False, formatters={"elasticidad": "{:9.4f}".format}))

analitica = (NOMINAL["Cinv"] * crf_nominal
             / (NOMINAL["Cinv"] * crf_nominal + NOMINAL["Com"]))
print()
ok = [comparar("elasticidad de la irradiación", elast["Hs"], -1.0, 1e-6, ""),
      comparar("elasticidad de la razón de desempeño", elast["PR"], -1.0,
               1e-6, ""),
      comparar("elasticidad de la inversión", elast["Cinv"],
               LIBRO["lcoe_elasticidad_inversion"], 5e-6, ""),
      comparar("elasticidad de la tasa de descuento", elast["i"],
               LIBRO["lcoe_elasticidad_tasa"], 5e-4, ""),
      comparar("valor analítico de la elasticidad de la inversión",
               analitica, LIBRO["lcoe_elasticidad_inversion"], 5e-7, ""),
      comparar("suma de las elasticidades de inversión y operación",
               elast["Cinv"] + elast["Com"], 1.0, 1e-6, "")]

if REVISAR_ELAST:
    assert all(ok), "las elasticidades no reproducen el Ejemplo 3.7"
    print("\nla irradiación y la razón de desempeño aparecen como factores "
          "del denominador, de modo que su elasticidad vale exactamente "
          "menos uno, y las dos componentes de costo suman la unidad "
          "porque entran de manera aditiva en el numerador")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_ELAST = True")

## 2. Un factor a la vez y el diagrama de tornado

La elasticidad describe solo el entorno inmediato del punto nominal. El
método de un factor a la vez recorre el rango plausible completo de cada
entrada mientras las demás permanecen fijas, y su representación es el
diagrama de tornado de la Figura 3.13, en el cual cada barra abarca el
intervalo de respuestas de una entrada y las barras se ordenan por
amplitud decreciente.

### Ejercicio 2

Complete el cálculo de la amplitud relativa que cada factor induce sobre
la respuesta al recorrer su rango plausible. La celda de partida usa un
rango de más y menos uno por ciento para todos los factores, con lo cual
el tornado solo repite el orden de las elasticidades.

In [ ]:
# COMPLETE: evalúe el modelo en el mínimo y en el máximo de cada factor
# y devuelva ambos valores junto con la amplitud relativa
#   amplitud = 100*|y(maximo) - y(minimo)|/y(nominal)
REVISAR_TORNADO = False


def tornado(modelo, nominal: dict, rangos: dict) -> pd.DataFrame:
    """Respuesta en los extremos del rango de cada factor."""
    y0 = modelo(**nominal)
    filas = []
    for factor, (bajo, alto) in rangos.items():
        izq, der = dict(nominal), dict(nominal)
        izq[factor] = nominal[factor] * 0.99
        der[factor] = nominal[factor] * 1.01
        y_izq, y_der = modelo(**izq), modelo(**der)
        filas.append((factor, y_izq, y_der,
                      100 * abs(y_der - y_izq) / y0))
    return pd.DataFrame(filas, columns=["factor", "en el mínimo",
                                        "en el máximo", "amplitud (%)"])

In [ ]:
barras = tornado(lcoe, NOMINAL, RANGOS).sort_values("amplitud (%)",
                                                    ascending=False)
barras["elasticidad"] = [elast[f] for f in barras["factor"]]
print(barras.to_string(index=False,
                       formatters={"en el mínimo": "{:.6f}".format,
                                   "en el máximo": "{:.6f}".format,
                                   "amplitud (%)": "{:6.1f}".format,
                                   "elasticidad": "{:9.4f}".format}))

ok = [comparar("amplitud de la tasa de descuento",
               float(barras.loc[barras["factor"] == "i", "amplitud (%)"].iloc[0]),
               LIBRO["lcoe_amplitud_tasa"], 5e-2, "por ciento"),
      comparar("amplitud de la irradiación",
               float(barras.loc[barras["factor"] == "Hs", "amplitud (%)"].iloc[0]),
               LIBRO["lcoe_amplitud_irradiacion"], 5e-2, "por ciento")]

if REVISAR_TORNADO:
    assert all(ok), "las amplitudes no reproducen la Figura 3.13"
    assert barras.iloc[0]["factor"] == "i", \
        "la tasa de descuento debe encabezar el tornado"
    print("\nel tornado invierte el orden que sugieren las elasticidades, "
          "porque la tasa de descuento tiene elasticidad 0.637 y un rango "
          "amplio, mientras que la irradiación tiene elasticidad menos uno "
          "y un rango estrecho")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_TORNADO = True")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.4))
orden = barras.iloc[::-1].reset_index(drop=True)
for j, fila in orden.iterrows():
    izq = min(fila["en el mínimo"], fila["en el máximo"])
    der = max(fila["en el mínimo"], fila["en el máximo"])
    ax.barh(j, costo_nominal - izq, left=izq, height=0.6,
            color=COLORES["azul"], alpha=0.85)
    ax.barh(j, der - costo_nominal, left=costo_nominal, height=0.6,
            color=COLORES["naranja"], alpha=0.9)
    ax.text(der + 0.0012, j, f"{fila['amplitud (%)']:.1f} por ciento",
            va="center", fontsize=8.2, color=COLORES["gris"])
ax.axvline(costo_nominal, color=COLORES["rojo"], lw=1.1)
ax.set_yticks(range(len(orden)))
etiquetas = {"Hs": "irradiación", "PR": "razón de desempeño",
             "Cinv": "inversión", "Com": "operación",
             "i": "tasa de descuento", "n": "vida útil",
             "delta": "degradación"}
ax.set_yticklabels([etiquetas[f] for f in orden["factor"]], fontsize=9)
ax.set_xlabel("costo nivelado de la energía (USD por kW h)")
ax.set_xlim(0.062, 0.114)
ax.grid(axis="y", visible=False)
ax.set_title(f"Diagrama de tornado, valor nominal {costo_nominal:.4f}")
plt.tight_layout()
plt.show()

## 3. Tamizado de Morris

Los métodos anteriores mueven un factor a la vez y recorren solo los ejes
que pasan por el punto nominal, de modo que no detectan interacciones ni
efectos que cambian con la posición. El Algoritmo 3.4 del libro construye
trayectorias sobre una malla del hipercubo unitario, en las cuales cada
factor se perturba una sola vez, y promedia los valores absolutos de los
cocientes incrementales. El promedio mide la influencia global del factor
y la desviación mide cuánto varía esa influencia según la región, de modo
que una desviación grande delata no linealidad o interacción. El costo es
de \(r(k+1)\) evaluaciones.

El modelo de aplicación vuelve a ser el biodigestor del Ejemplo 3.6, con
sus seis entradas y sus distribuciones.

In [ ]:
from scipy.stats import lognorm, norm, qmc, triang, uniform

PCI = 35.8
FACTORES = ["m", "SV", "B0", "k", "TRH", "eta"]


def transformar(u: np.ndarray) -> dict:
    """Del hipercubo unitario a las variables físicas del biodigestor."""
    s = np.sqrt(np.log(1 + 0.18**2))
    return dict(m=norm.ppf(u[:, 0], 1200.0, 90.0),
                SV=norm.ppf(u[:, 1], 0.115, 0.012),
                B0=lognorm.ppf(u[:, 2], s, scale=0.21),
                k=uniform.ppf(u[:, 3], 0.10, 0.15),
                TRH=triang.ppf(u[:, 4], 6 / 13, loc=22.0, scale=13.0),
                eta=triang.ppf(u[:, 5], 0.5, loc=0.28, scale=0.08))


def energia_unitaria(u: np.ndarray) -> np.ndarray:
    """Energía eléctrica diaria evaluada sobre el hipercubo unitario."""
    p = transformar(u)
    return (p["m"] * p["SV"] * p["B0"] * (1 - np.exp(-p["k"] * p["TRH"]))
            * PCI * p["eta"] / 3.6)

### Ejercicio 3

Complete los efectos elementales del tamizado. Para cada trayectoria, el
efecto elemental del factor recorrido vale la diferencia de respuestas
dividida por el paso, y al terminar se promedian los valores absolutos
sobre las trayectorias. La celda de partida devuelve ceros. El diseño de
las trayectorias ya está escrito, incluido el margen que mantiene los
puntos dentro del hipercubo abierto, necesario porque dos de las
marginales son normales y su función cuantílica no está definida en cero
ni en uno.

In [ ]:
# COMPLETE: efectos[t, j] = (y[t, pos+1] - y[t, pos])/delta
REVISAR_MORRIS = False


def morris(modelo, k: int, r: int = 30, p: int = 8,
           semilla: int = SEMILLA, margen: float = 1e-3):
    """Tamizado por efectos elementales sobre trayectorias."""
    generador = np.random.default_rng(semilla)
    delta = p / (2 * (p - 1))
    niveles = np.arange(p // 2) / (p - 1)
    puntos, ordenes = [], []
    for _ in range(r):
        x = generador.choice(niveles, size=k)
        orden = generador.permutation(k)
        trayectoria = [x.copy()]
        for j in orden:
            x = x.copy()
            x[j] += delta
            trayectoria.append(x.copy())
        puntos.append(np.array(trayectoria))
        ordenes.append(orden)
    diseno = np.vstack(puntos)
    y = modelo(margen + (1 - 2 * margen) * diseno).reshape(r, k + 1)
    efectos = np.zeros((r, k))
    for t in range(r):
        for posicion, j in enumerate(ordenes[t]):
            efectos[t, j] = 0.0        # marcador de posición
    return (np.abs(efectos).mean(axis=0), efectos.std(axis=0, ddof=1),
            diseno.shape[0])

La verificación de una implementación de este tipo se hace con un modelo
cuyos efectos se conocen de antemano. Sobre un modelo lineal
\(g(\mathbf{u}) = \sum_j a_j u_j\), cada efecto elemental vale
exactamente \(a_j\), de modo que el promedio de sus valores absolutos
debe reproducir \(|a_j|\) y la desviación debe anularse.

In [ ]:
COEFICIENTES = np.array([4.0, -2.0, 1.0, 0.0])
lineal = lambda u: u @ COEFICIENTES

mu_lin, sigma_lin, ev_lin = morris(lineal, k=4, r=12, margen=0.0)
esperado = np.abs(COEFICIENTES)
print("verificación con un modelo lineal de coeficientes conocidos")
print(pd.DataFrame({"coeficiente": COEFICIENTES, "mu estrella": mu_lin,
                    "sigma": sigma_lin}).to_string(
    index=False, formatters={"mu estrella": "{:9.6f}".format,
                             "sigma": "{:9.2e}".format}))

if REVISAR_MORRIS:
    assert np.allclose(mu_lin, esperado, atol=1e-10), \
        "sobre un modelo lineal mu estrella debe valer el coeficiente"
    assert np.allclose(sigma_lin, 0.0, atol=1e-10), \
        "sobre un modelo lineal la desviación de los efectos debe anularse"
    print("\nla implementación reproduce los efectos exactos del modelo lineal")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_MORRIS = True")

In [ ]:
mu_bio, sigma_bio, ev_bio = morris(energia_unitaria, k=6, r=30, p=8)
tamizado = pd.DataFrame({"factor": FACTORES, "mu estrella (kW h/d)": mu_bio,
                         "sigma (kW h/d)": sigma_bio})
tamizado = tamizado.sort_values("mu estrella (kW h/d)", ascending=False)
print(tamizado.to_string(index=False,
                         formatters={"mu estrella (kW h/d)": "{:8.2f}".format,
                                     "sigma (kW h/d)": "{:8.2f}".format}))

jerarquia = list(tamizado["factor"])
comparar("evaluaciones del tamizado", ev_bio, LIBRO["morris_evaluaciones"],
         0, "")
print(f"\njerarquía obtenida   {jerarquia}")
print("jerarquía del libro  ['B0', 'SV', 'm', 'eta', 'k', 'TRH']")

if REVISAR_MORRIS:
    assert ev_bio == 210, "treinta trayectorias de seis factores cuestan 210 evaluaciones"
    assert jerarquia == ["B0", "SV", "m", "eta", "k", "TRH"], \
        "la jerarquía no coincide con la que publica la sección 3.6.2"
    print("la jerarquía y el costo coinciden con la sección 3.6.2 del libro")

Los valores de mu estrella que aquí resultan no coinciden cifra a cifra
con los que publica el libro, que valen 58.24, 42.55, 33.65, 22.40, 3.93
y 2.56 kW h/d. La razón es que mu estrella depende del diseño concreto de
las treinta trayectorias y del tratamiento de los bordes del hipercubo,
que el libro no declara, y con treinta trayectorias su error de muestreo
supera el diez por ciento. Lo que el método promete y aquí se verifica es
la jerarquía, que coincide de manera exacta, y el costo de 210
evaluaciones. El propio libro advierte que el tamizado descarta y
jerarquiza mientras que el análisis basado en varianza cuantifica.

## 4. Sensibilidad global basada en varianza

El Teorema 3.4 del libro descompone la varianza de la respuesta en
contribuciones de cada subconjunto de factores, y la Definición 3.10
resume esa descomposición en dos números por factor. El índice de primer
orden es la fracción de la varianza que se eliminaría si el factor
quedara fijo en su valor verdadero, y el índice total es la fracción que
permanecería si todos los demás quedaran fijos, de modo que recoge el
efecto individual más todas las interacciones.

### Ejercicio 4

Complete los dos estimadores del Listado 3.10, el de Saltelli para el
primer orden y el de Jansen para el total. El primero promedia el
producto de la respuesta en la matriz B por la diferencia entre la
respuesta en la matriz cruzada y la respuesta en A, y el segundo promedia
el cuadrado de la diferencia entre A y la cruzada, dividido por dos veces
la varianza. La celda de partida devuelve un primer orden nulo y un total
unitario.

In [ ]:
# COMPLETE: S[j]  = mean(yB*(yAB - yA))/V          estimador de Saltelli
#           ST[j] = mean((yA - yAB)**2)/(2*V)      estimador de Jansen
REVISAR_SOBOL = False


def indices_sobol(modelo, d: int, m: int, semilla: int = SEMILLA):
    """Índices de primer orden y totales por muestreo cruzado."""
    Z = qmc.Sobol(d=2 * d, scramble=True, seed=semilla).random(2**m)
    A, B = Z[:, :d], Z[:, d:]
    yA, yB = modelo(A), modelo(B)
    V = np.var(np.concatenate([yA, yB]), ddof=1)
    S, ST = np.empty(d), np.empty(d)
    for j in range(d):
        AB = A.copy()
        AB[:, j] = B[:, j]
        yAB = modelo(AB)
        S[j] = 0.0        # marcador de posición
        ST[j] = 1.0       # marcador de posición
    return S, ST

Una implementación así debe verificarse antes de aplicarla, porque un
error de índice o de signo produce resultados plausibles. La verificación
estándar emplea la función de prueba de Ishigami, cuyos índices se
conocen en forma cerrada y valen 0.3139, 0.4424 y cero para el primer
orden, con un índice total de 0.2437 para el tercer factor, que no tiene
efecto individual y actúa solo por interacción.

In [ ]:
def ishigami(u: np.ndarray) -> np.ndarray:
    """Función de prueba de Ishigami sobre el hipercubo unitario."""
    x = -np.pi + 2 * np.pi * u
    return (np.sin(x[:, 0]) + 7 * np.sin(x[:, 1])**2
            + 0.1 * x[:, 2]**4 * np.sin(x[:, 0]))


S_ish, ST_ish = indices_sobol(ishigami, d=3, m=14)
exactos_S = np.array([0.3139, 0.4424, 0.0])
exactos_ST = np.array([0.5576, 0.4424, 0.2437])
print(pd.DataFrame({"factor": ["x1", "x2", "x3"], "S estimado": S_ish,
                    "S exacto": exactos_S, "ST estimado": ST_ish,
                    "ST exacto": exactos_ST}).to_string(
    index=False, float_format=lambda v: f"{v:9.4f}"))

ok = [comparar("primer orden de x1", S_ish[0], LIBRO["ishigami_s1"], 5e-5, ""),
      comparar("primer orden de x2", S_ish[1], LIBRO["ishigami_s2"], 5e-5, ""),
      comparar("primer orden de x3", S_ish[2], LIBRO["ishigami_s3"], 5e-5, ""),
      comparar("índice total de x3", ST_ish[2], LIBRO["ishigami_st3"], 5e-5, "")]

if REVISAR_SOBOL:
    assert all(ok), "los índices no reproducen la verificación de la sección 3.6.2"
    assert np.max(np.abs(S_ish - exactos_S)) < 5e-4, \
        "el primer orden debe coincidir con los valores cerrados"
    print("\nlos estimadores reproducen los índices exactos de Ishigami")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_SOBOL = True")

In [ ]:
S_bio, ST_bio = indices_sobol(energia_unitaria, d=6, m=14)
indices = pd.DataFrame({"factor": FACTORES, "primer orden": S_bio,
                        "total": ST_bio})
indices["interaccion"] = indices["total"] - indices["primer orden"]
print(indices.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print(f"\nsuma de los índices de primer orden   {S_bio.sum():.3f}")
print(f"peso de las interacciones             {100 * (1 - S_bio.sum()):.1f} por ciento")
print(f"evaluaciones del modelo               {(6 + 2) * 2**14:,d}")

ok = [comparar("primer orden del potencial metanogénico", S_bio[2],
               LIBRO["sobol_biogas_s_B0"], 5e-4, ""),
      comparar("suma de los índices de primer orden", S_bio.sum(),
               LIBRO["sobol_biogas_suma_primer_orden"], 5e-4, "")]
publicados_S = np.array([0.107, 0.207, 0.616, 0.005, 0.001, 0.050])
publicados_ST = np.array([0.112, 0.216, 0.629, 0.005, 0.001, 0.052])
print("\nprimer orden publicado ", publicados_S)
print("primer orden calculado ", np.round(S_bio, 3))
print("total publicado        ", publicados_ST)
print("total calculado        ", np.round(ST_bio, 3))
print("el índice total del potencial metanogénico sale 0.628 y el libro "
      "publica 0.629, diferencia de una milésima en la tercera cifra")

if REVISAR_SOBOL:
    assert np.max(np.abs(np.round(S_bio, 3) - publicados_S)) < 5e-4, \
        "los índices de primer orden deben coincidir con los publicados"
    assert np.max(np.abs(np.round(ST_bio, 3) - publicados_ST)) < 1.5e-3, \
        "los índices totales deben coincidir con los publicados"
    print("los índices reproducen la sección 3.6.2 del libro")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
posicion = np.arange(len(FACTORES))
ax1.barh(posicion - 0.18, S_bio, height=0.34, color=COLORES["azul"],
         label="primer orden")
ax1.barh(posicion + 0.18, ST_bio, height=0.34, color=COLORES["naranja"],
         label="total")
ax1.set_yticks(posicion)
ax1.set_yticklabels(FACTORES)
ax1.set_xlabel("fracción de la varianza de la respuesta")
ax1.set_title("Índices de Sobol' del biodigestor")
ax1.grid(axis="y", visible=False)
ax1.legend()

ax2.plot(mu_bio, sigma_bio, "o", color=COLORES["azul"], ms=6)
for f, x, y in zip(FACTORES, mu_bio, sigma_bio):
    ax2.annotate(f, (x, y), textcoords="offset points", xytext=(6, 4),
                 fontsize=9, color=COLORES["gris"])
tope = float(max(mu_bio.max(), sigma_bio.max())) * 1.15
ax2.plot([0, tope], [0, tope], "--", color=COLORES["rojo"], lw=0.9)
ax2.set_xlabel("mu estrella (kW h/d)")
ax2.set_ylabel("sigma de los efectos elementales (kW h/d)")
ax2.set_title("Plano de Morris, influencia y no linealidad")
plt.tight_layout()
plt.show()

## 5. Qué se hace con el resultado

Queda por decidir qué se hace con la jerarquía, que es la parte con
consecuencias reales. Los factores dominantes exigen inversión en
medición, en calibración o en control. Los irrelevantes se fijan en su
valor nominal y se declara esa decisión, lo cual simplifica el modelo y
reduce la recolección de datos. Los intermedios se reportan tal como
están.

### Ejercicio 5

Complete la clasificación de los factores a partir del índice total. Un
factor con índice total por encima del umbral alto se marca para medir
mejor, uno por debajo del umbral bajo se fija en su valor nominal y el
resto se reporta. La celda de partida marca todo como intermedio.

In [ ]:
# COMPLETE: asigne la decisión según el índice total de cada factor.
REVISAR_DECISION = False


def clasificar(indices: pd.DataFrame, alto: float = 0.10,
               bajo: float = 0.01) -> pd.DataFrame:
    """Traduce los índices totales en una decisión de medición."""
    tabla = indices.copy()
    tabla["decision"] = "reportar tal como está"
    return tabla.sort_values("total", ascending=False)

In [ ]:
decision = clasificar(indices)
print(decision.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

dominante = decision.iloc[0]
print(f"\nel factor dominante es {dominante['factor']} y explica el "
      f"{100 * dominante['primer orden']:.0f} por ciento de la varianza")
fijables = list(decision.loc[decision["decision"].str.startswith("fijar"),
                             "factor"])
print(f"factores que pueden fijarse en su valor nominal   {fijables}")

if REVISAR_DECISION and REVISAR_SOBOL:
    assert dominante["factor"] == "B0", \
        "el potencial metanogénico debe encabezar la jerarquía"
    assert abs(100 * dominante["primer orden"] - 62) < 1, \
        "el libro reporta que el potencial metanogénico explica el 62 por ciento"
    assert set(fijables) == {"k", "TRH"}, \
        "la constante de hidrólisis y el tiempo de retención pueden fijarse"
    print("\nla recomendación coincide con la del libro, medir el potencial "
          "metanogénico con un ensayo específico y fijar el tiempo de "
          "retención y la constante de hidrólisis")
else:
    print("\ncomplete las celdas anteriores y ponga las banderas en True")

La comparación entre los tres enfoques queda a la vista. El tamizado de
Morris gastó 210 evaluaciones y entregó la misma jerarquía que el análisis
de Sobol', que gastó 131072, es decir seiscientas veces más. El tamizado
descarta y jerarquiza mientras que el basado en varianza cuantifica, y la
estrategia sensata los encadena, tamizando primero para descartar y
cuantificando después sobre los que sobrevivieron. La Tabla 3.4 del libro
resume el costo, el supuesto y la limitación de cada familia.

In [ ]:
enfoques = pd.DataFrame(
    [("elasticidad local", 2 * len(NOMINAL), "modelo suave en el punto",
      "no linealidad e interacción"),
     ("un factor a la vez", 2 * len(RANGOS), "efectos separables",
      "interacción entre factores"),
     ("tamizado de Morris", ev_bio, "nada sobre la forma",
      "el reparto de la varianza"),
     ("índices de Sobol'", (6 + 2) * 2**14, "factores independientes",
      "nada relevante")],
    columns=["enfoque", "evaluaciones", "qué supone", "qué no detecta"])
print(enfoques.to_string(index=False))
print(f"\nrazón de costo entre Sobol' y Morris   "
      f"{(6 + 2) * 2**14 / ev_bio:.0f} veces")

## 6. Problemas del capítulo

El problema 3-23 presenta un factor con elasticidad menos 1.8 y rango del
3 por ciento y otro con elasticidad 0.4 y rango del 40 por ciento, y pide
determinar cuál domina el tornado y por qué la elasticidad sola induce a
error. El problema 3-27 pide implementar el tamizado de Morris y
verificarlo, tarea que ya quedó resuelta arriba con el modelo lineal.

In [ ]:
candidatos = pd.DataFrame(
    [("factor A", -1.8, 3.0), ("factor B", 0.4, 40.0)],
    columns=["factor", "elasticidad", "rango relativo (%)"])
candidatos["amplitud aproximada (%)"] = (candidatos["elasticidad"].abs()
                                         * candidatos["rango relativo (%)"])
print(candidatos.to_string(index=False))
print("\nel factor B domina el tornado pese a su elasticidad menor, porque "
      "la amplitud es el producto de la elasticidad por el rango realmente "
      "alcanzable, y la elasticidad sola ignora cuánto puede moverse cada "
      "entrada")

print("\nen el sistema fotovoltaico ocurre lo mismo")
print(barras[["factor", "elasticidad", "amplitud (%)"]].to_string(
    index=False, formatters={"elasticidad": "{:9.4f}".format,
                             "amplitud (%)": "{:6.1f}".format}))

## 7. Cierre

Al terminar este cuaderno el estudiante debe poder hacer lo siguiente.

1. Calcular elasticidades por diferencias centradas y usar las
   identidades estructurales del modelo como verificación gratuita. Un
   factor multiplicativo debe dar elasticidad uno y uno del denominador
   menos uno. Revise la Definición 3.9 si no ocurre.
2. Construir un tornado y explicar por qué su orden difiere del de las
   elasticidades. Revise el comentario del Ejemplo 3.7.
3. Implementar el tamizado de Morris, verificarlo con un modelo de
   efectos conocidos y leer el plano de influencia contra no linealidad.
   Revise el Algoritmo 3.4.
4. Estimar los índices de Sobol' de primer orden y totales, verificarlos
   con Ishigami y leer la diferencia entre ambos como medida de
   interacción. Revise el Teorema 3.4 y la Definición 3.10.
5. Traducir la jerarquía en una decisión de medición, de calibración o de
   simplificación, y declararla en el informe.

El cuaderno siguiente reúne todo lo anterior y se ocupa de la discusión
de los resultados en función del problema de estudio.